In [5]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from statistics import mode

API_KEY = "e8ce120f8c0f4cd0b3d80608252105"
BASE_URL = "http://api.weatherapi.com/v1/history.json"

location = "Danang, Vietnam"
start_date = datetime(2024, 5, 22)
end_date = datetime(2025, 5, 22)

In [6]:
def get_weather_data(location, date):
    url = f"{BASE_URL}?key={API_KEY}&q={location}&dt={date}"
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        return data['forecast']['forecastday'][0]['hour']
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e}")
        print(f"Response content: {response.text}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Lỗi khi lấy dữ liệu cho {location} ngày {date}: {e}")
        return None

def crawl_weather_data_hourly(location, start_date, end_date):
    data_list = []
    current_date = start_date
    while current_date <= end_date:
        print(f"Đang crawl dữ liệu cho ngày: {current_date.strftime('%Y-%m-%d')}")
        hourly_data = get_weather_data(location, current_date.strftime('%Y-%m-%d'))
        if hourly_data:
            for hour_data in hourly_data:
                try:
                    data_hour = {
                        'time': hour_data['time'],                  # thời gian
                        'temp_c': hour_data['temp_c'],              # nhiệt độ (C)
                        'precip_mm': hour_data['precip_mm'],        # lượng mưa (mm)
                        'humidity': hour_data['humidity'],          # độ ẩm (%)
                        'pressure_mb': hour_data['pressure_mb'],    # áp suất (mb)
                        'wind_kph': hour_data['wind_kph'],          # tốc độ gió (kph)
                        'cloud': hour_data['cloud'],                # độ che phủ mây (%)
                        'dewpoint_c': hour_data['dewpoint_c'],      # điểm sương (C)
                        'feelslike_c': hour_data['feelslike_c'],    # nhiệt độ cảm nhận (C)
                        'vis_km': hour_data['vis_km'],              # tầm nhìn (km)
                        'gust_kph': hour_data['gust_kph'],          # gió giật (kph)
                        'wind_dir': hour_data['wind_dir'],          # hướng gió
                        'condition': hour_data['condition']['text'] # điều kiện thời tiết
                    }
                    data_list.append(data_hour)
                except KeyError as e:
                    print(f"Lỗi khi xử lý dữ liệu cho giờ {hour_data.get('time', 'unknown')}: {e}")
        time.sleep(0.01)
        current_date += timedelta(days=1)
    
    return data_list

weather_data = crawl_weather_data_hourly(location, start_date, end_date)

if weather_data:
    df = pd.DataFrame(weather_data)
    df['dewpoint_depression'] = df['temp_c'] - df['dewpoint_c']
    df.to_csv('weather_data.csv', index=False)
    print(f"Đã crawl được {len(df)} mẫu và đã được lưu vào weather_data.csv")
else:
    print("Không lấy được dữ liệu.")

Đang crawl dữ liệu cho ngày: 2024-05-22
HTTP Error: 400 Client Error: Bad Request for url: http://api.weatherapi.com/v1/history.json?key=e8ce120f8c0f4cd0b3d80608252105&q=Danang,%20Vietnam&dt=2024-05-22
Response content: {"error":{"code":1008,"message":"API key is limited to get history data. Please check our pricing page and upgrade to higher plan."}}
Đang crawl dữ liệu cho ngày: 2024-05-23
Đang crawl dữ liệu cho ngày: 2024-05-24
Đang crawl dữ liệu cho ngày: 2024-05-25
Đang crawl dữ liệu cho ngày: 2024-05-26
Đang crawl dữ liệu cho ngày: 2024-05-27
Đang crawl dữ liệu cho ngày: 2024-05-28
Đang crawl dữ liệu cho ngày: 2024-05-29
Đang crawl dữ liệu cho ngày: 2024-05-30
Đang crawl dữ liệu cho ngày: 2024-05-31
Đang crawl dữ liệu cho ngày: 2024-06-01
Đang crawl dữ liệu cho ngày: 2024-06-02
Đang crawl dữ liệu cho ngày: 2024-06-03
Đang crawl dữ liệu cho ngày: 2024-06-04
Đang crawl dữ liệu cho ngày: 2024-06-05
Đang crawl dữ liệu cho ngày: 2024-06-06
Đang crawl dữ liệu cho ngày: 2024-06-07
Đang c

In [10]:
df.loc[df['feelslike_c'] <= 22, 'feelslike_c'] += 2

In [ ]:
from sklearn.model_selection import train_test_split

# Tách theo tỉ lệ 80% - 20%
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

In [13]:
df_train.to_csv('raw_data_train.csv', index=False)
df_test.to_csv('raw_data_test.csv', index=False)
